In [11]:
using XGPaint
using CSV, DataFrames
using Healpix
using Interpolations
using JLD2
using Dates
using Printf

# ──────────────────────────────────────────────────────────────────────────────
# Fixed cosmology / model parameters (same as before)
# ──────────────────────────────────────────────────────────────────────────────
const h       = 0.6766
const Ob0h2   = 0.02242
const Oc0h2   = 0.1193
const Omega_b = Ob0h2 / h^2
const Omega_c = Oc0h2 / h^2
const B       = 1.41

# 10 arcmin beam (radians, plain Float64)
const θ_FWHM  = 10.0 * (π / 180) / 60

# Cache file for the 2D beamed interpolator
# const CACHE_FILE = "cached_a10_beamed2d_B1p41_highpres.jld2"
const CACHE_FILE = "cached_a10_beamed2d_B1p41_ultrahighpres.jld2"

"cached_a10_beamed2d_B1p41_ultrahighpres.jld2"

In [12]:
using XGPaint
using Unitful  # for uconvert

# your fixed params
const h       = 0.6766
const Ob0h2   = 0.02242
const Oc0h2   = 0.1193
const Omega_b = Ob0h2 / h^2
const Omega_c = Oc0h2 / h^2
const B       = 1.41

# build the model (this sets model.cosmo internally)
model = XGPaint.Arnauld10ThermalSZProfile(
    Omega_c = Omega_c,
    Omega_b = Omega_b,
    h       = h,
    B       = B
)

# evaluate ρ_crit at your redshift
z = 0.0
rho_c = XGPaint.ρ_crit(model, z)                     # in kg/m^3
rho_c

8.598814256622896e-27 kg m⁻³

In [13]:
function build_beam_convolved_A10_2D(a10_base::Arnauld10ThermalSZProfile{T};
                                     θ_FWHM::Real,
                                     N_logθ::Int=256, pad::Int=128,
                                     N_logθ500::Int=128,
                                     logθ_min::Real=-16.5,  logθ_max::Real=2.5,
                                     logθ500_min::Real=-9.7, logθ500_max::Real=0.0) where T
    # 1) Universal shape F(u) spline
    shape = A10ThetaProfile(a10_base; Nx=1024, x_min=1e-12, x_max=1e12)

    # 2) 2D grid F(θ, θ500)
    logθs, logθ500s, A = profile_grid_θ_θ500(shape;
        N_logθ=N_logθ, N_logθ500=N_logθ500,
        logθ_min=logθ_min, logθ_max=logθ_max,
        logθ500_min=logθ500_min, logθ500_max=logθ500_max)

    lrange = [1/exp(2.5), 1/exp(-16.5)]
    rrange = [exp(-16.5),  exp(2.5)]
    rft    = RadialFourierTransform(n   = 4096,
                                    pad = 2048,
                                    lrange = lrange,
                                    rrange = rrange)

    # 3) Beam on θ via RFT (same convention as your old code)
    # rft    = RadialFourierTransform(n=N_logθ, pad=pad)
    lbeam  = exp.(- (rft.l .* (rft.l .+ 1)) .* (θ_FWHM^2) ./ (16*log(2)))
    transform_profile_grid_θ!(A, rft, lbeam)
    cleanup_negatives_θ!(A)

    # 4) 2D cubic spline over (logθ, logθ500) with zero extrapolation
    itp_core = Interpolations.interpolate(
        A, Interpolations.BSpline(Interpolations.Cubic(Interpolations.Line(Interpolations.OnGrid()))))
    itp2d = Interpolations.extrapolate(Interpolations.scale(itp_core, logθs, logθ500s), 0.0)

    return BeamConvolvedA10_2D(itp2d, a10_base)
end


build_beam_convolved_A10_2D (generic function with 1 method)

In [14]:
# ──────────────────────────────────────────────────────────────────────────────
# Build/load the 2D beamed interpolator: F_beam(logθ, logθ500)
# (uses XGPaint.build_beam_convolved_A10_2D)
# ──────────────────────────────────────────────────────────────────────────────
function get_beamed_interpolator_2d(cache_file::String)
    if isfile(cache_file)
        @info "Loading 2D beamed interpolator from cache: $cache_file"
        return JLD2.load(cache_file, "y_model_beamed2d")
    else
        @info "Cache not found; building 2D beamed interpolator…"
        y_base = Arnauld10ThermalSZProfile(Omega_c = Omega_c,
                                           Omega_b = Omega_b,
                                           h       = h,
                                           B       = B)

        # Build with the same θ grid convention as your original (N_logθ=256, pad=128)
        y_model_beamed2d = build_beam_convolved_A10_2D(y_base; θ_FWHM=θ_FWHM,
                                                       N_logθ=4096, pad=2048,
                                                       N_logθ500=8192,
                                                       logθ500_min=-9.7, logθ500_max=3.0)
        mkpath(dirname(cache_file))
        @save cache_file y_model_beamed2d
        @info "Saved → $cache_file"
        return y_model_beamed2d
    end
end

const GLOBAL_T0 = now()
const y_model_beamed2d = CachedBeamConvolvedA10_2D(get_beamed_interpolator_2d(CACHE_FILE))
@info "Interpolator ready."

# ──────────────────────────────────────────────────────────────────────────────
# Paint one catalogue to a Healpix map and save to FITS (same logic)
# ──────────────────────────────────────────────────────────────────────────────
function process_catalogue_signal(catalogue_path::String, output_map_path::String)
    @info "Processing: $catalogue_path"

    # ------------------------ Read halo catalogue ---------------------------
    df         = CSV.read(catalogue_path, DataFrame)
    redshift   = df.z
    halo_mass  = df.M .* 1e14  # convert to M☉
    ra         = rem.(df.lon .+ π, 2π) .- π
    dec        = df.lat .- π/2
    snr = df.snr

    # --- Apply cuts: keep only z > 0.2 and M < 1e15 M_sun ---
    mask = (snr .< 5.) 
    redshift = redshift[mask]
    halo_mass = halo_mass[mask]
    ra       = ra[mask]
    dec      = dec[mask]


    nside       = 1024
    θmax_deg    = 5.0
    cluster_map = HealpixMap{Float64,RingOrder}(nside)
    workspace   = HealpixProfileWorkspace(nside, deg2rad(θmax_deg))

    paint!(cluster_map, workspace, y_model_beamed2d, halo_mass, redshift, ra, dec)

    if isfile(output_map_path)
        rm(output_map_path; force=true)
    end
    Healpix.saveToFITS(cluster_map, output_map_path, typechar="D")
    @info "Saved → $output_map_path"
end

[ Info: Cache not found; building 2D beamed interpolator…
[ Info: Saved → cached_a10_beamed2d_B1p41_ultrahighpres.jld2
[ Info: Interpolator ready.


process_catalogue_signal (generic function with 1 method)

In [15]:
# for i in 0:1
#     catalogue_path = "../catalogue_demo/catalogue_sbi_$(i).csv"
#     output_map_path = "../maps_fullsky/map_beam10arcmin_nside512_$(i).fits"
#     mkpath(dirname(output_map_path))                # ensure directory exists
#     process_catalogue_signal(catalogue_path, "!" * output_map_path)  # clobber if exists
# end


In [16]:
i = 0
catalogue_path = "../catalogue_sbi_snr_$(i).csv"
output_map_path = "../map_beam10arcmin_masked_nside1024_snr5_$(i).fits"
mkpath(dirname(output_map_path))                # ensure directory exists
process_catalogue_signal(catalogue_path, "!" * output_map_path)  # clobber if exists



[ Info: Processing: ../catalogue_sbi_snr_0.csv
[ Info: Saved → !../map_beam10arcmin_masked_nside1024_snr5_0.fits
